# 02 — Stage A Baseline Classifier (MobileNetV2 Transfer Learning & Grad-CAM)
**Project:** RoadWatch — Pothole Detection from Road Images using Deep Learning

### Methodology (Section 8.1)
- Pretrained **MobileNetV2** (`IMAGENET1K_V1` weights) with classification head replaced by 2 classes (`0: normal`, `1: pothole`).
- **Step 1:** Freeze backbone and train classification head for 5–10 epochs.
- **Step 2:** Unfreeze final inverted-residual blocks and fine-tune at `lr=1e-4` with Adam and Cross-Entropy loss.
- Evaluate Accuracy, Precision, Recall, F1-score, Confusion Matrix, and ROC-AUC, then export `../models/classifier_mobilenetv2.pt`.

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, roc_auc_score

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)

In [ ]:
def build_mobilenetv2_classifier(num_classes: int = 2, freeze_backbone: bool = True) -> nn.Module:
    model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
    if freeze_backbone:
        for param in model.features.parameters():
            param.requires_grad = False
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model

train_transforms = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.RandomRotation(degrees=8),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transforms = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])